# Day 4 — Web Scraping + Advanced Retrieval

**Goal:** extend the corpus with scraped data, upgrade retrieval to hybrid +
reranking, and introduce evaluation metrics beyond recall@k.

Morning: scrape responsibly. Afternoon: hybrid search (BM25 + dense) with
reciprocal rank fusion and cross-encoder reranking. End of day: Ragas for
generation quality metrics.

## 1. Setup

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
import time
import urllib.parse
import urllib.robotparser
from pathlib import Path

import requests
from bs4 import BeautifulSoup
import trafilatura
import numpy as np

from utils import (
    Chunk, make_chunk_id, add_chunks,
    get_chroma_client, get_llm_client,
)
from sentence_transformers import SentenceTransformer

chroma = get_chroma_client("./chroma_db")
client, generate = get_llm_client()
st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

## 2. Scraping ethics — robots.txt and rate limits

**Before writing a single line of scraper code, ALWAYS check robots.txt.**
A consultant whose scraper gets their client sued or IP-banned is a
consultant who doesn't come back to the engagement.

In [ ]:
def can_fetch(url: str, user_agent: str = "AcmeConsultingBot/1.0") -> bool:
    """Check robots.txt for permission to fetch a URL."""
    parsed = urllib.parse.urlparse(url)
    robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"
    rp = urllib.robotparser.RobotFileParser()
    rp.set_url(robots_url)
    try:
        rp.read()
    except Exception as e:
        print(f"Warning: could not read {robots_url}: {e}")
        return False
    return rp.can_fetch(user_agent, url)


# Our scraping target: Python.org's events page.
# python.org has a permissive robots.txt and publishes a public events feed.
TARGET = "https://www.python.org/events/python-events/"
print(f"Can we fetch {TARGET}? → {can_fetch(TARGET)}")

### Rate limiter

Be a polite client. 1 request per 2 seconds is a reasonable default for
unknown targets. For APIs with documented limits, use those instead.

In [ ]:
class RateLimiter:
    def __init__(self, min_interval_sec: float = 2.0):
        self.min_interval = min_interval_sec
        self.last_call = 0.0

    def wait(self):
        elapsed = time.time() - self.last_call
        if elapsed < self.min_interval:
            time.sleep(self.min_interval - elapsed)
        self.last_call = time.time()


limiter = RateLimiter(2.0)


HEADERS = {
    # Identify yourself. Some sites block unidentified bots outright.
    "User-Agent": "AcmeConsultingBot/1.0 (+mailto:consultants@example.com)"
}

## 3. Scrape the events listing

In [ ]:
def fetch(url: str) -> str:
    limiter.wait()
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()
    return r.text


listing_html = fetch(TARGET)
soup = BeautifulSoup(listing_html, "html.parser")

# python.org marks events with <ul class="list-recent-events"> and child <li>s
events = []
for li in soup.select("ul.list-recent-events li"):
    title_tag = li.find("h3")
    if not title_tag:
        continue
    link_tag = title_tag.find("a")
    title = title_tag.get_text(strip=True)
    href = link_tag["href"] if link_tag and link_tag.has_attr("href") else None

    time_tag = li.find("time")
    date = time_tag.get("datetime") if time_tag else None

    location_tag = li.find("span", class_="event-location")
    location = location_tag.get_text(strip=True) if location_tag else None

    events.append({
        "title": title,
        "url": urllib.parse.urljoin(TARGET, href) if href else None,
        "date": date,
        "location": location,
    })

print(f"Found {len(events)} events. First 3:")
for e in events[:3]:
    print(f"  {e['date']:<30s} {e['title']}  @ {e['location']}")

### Fetch each event's detail page

Use trafilatura to extract clean body text (strips nav, ads, boilerplate).

In [ ]:
def fetch_event_body(url: str) -> str:
    raw = fetch(url)
    body = trafilatura.extract(raw, include_comments=False, include_tables=False)
    return body or ""


# Limit to first 5 to keep notebook fast — in production you'd scrape all
event_chunks: list[Chunk] = []
for idx, ev in enumerate(events[:5]):
    if not ev["url"]:
        continue
    print(f"Fetching {ev['title']}...")
    body = fetch_event_body(ev["url"])
    text = f"EVENT: {ev['title']}\nDATE: {ev['date']}\nLOCATION: {ev['location']}\n\n{body[:2000]}"
    event_chunks.append(Chunk(
        id=make_chunk_id(ev["url"], 0, text),
        text=text,
        source=ev["url"],
        content_type="prose",
        document_type="event",
        extra={
            "event_title": ev["title"] or "",
            "event_date": ev["date"] or "",
            "event_location": ev["location"] or "",
            "scrape_timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
        },
    ))

print(f"\nScraped {len(event_chunks)} event records.")

## 4. Build the combined corpus

Merge the PEP chunks (from Day 3) with the scraped events. A real engagement
often looks like this — internal docs + scraped intel in one store.

In [ ]:
import re


def chunk_recursive(text, source, max_len=1500):
    def _split(t, seps):
        if len(t) <= max_len:
            return [t]
        for sep in seps:
            if sep == "":
                return [t[i:i + max_len] for i in range(0, len(t), max_len)]
            if sep in t:
                parts = t.split(sep)
                out, buf = [], ""
                for p in parts:
                    cand = (buf + sep + p) if buf else p
                    if len(cand) <= max_len:
                        buf = cand
                    else:
                        if buf:
                            out.append(buf)
                        if len(p) > max_len:
                            out.extend(_split(p, seps[1:]))
                            buf = ""
                        else:
                            buf = p
                if buf:
                    out.append(buf)
                return out
        return [t]

    pieces = _split(text, ["\n\n", "\n", ". ", " ", ""])
    return [
        Chunk(id=make_chunk_id(source, i, p), text=p, source=source,
              content_type="prose", document_type="pep",
              extra={"chunk_idx": i})
        for i, p in enumerate(pieces)
    ]


peps = []
for p in sorted(Path("./datasets/day1").glob("*.md")):
    pep_chunks = chunk_recursive(p.read_text(), p.name)
    peps.extend(pep_chunks)

corpus = peps + event_chunks
print(f"Combined corpus: {len(peps)} PEP chunks + {len(event_chunks)} event chunks = {len(corpus)} total")


try:
    chroma.delete_collection("day4_corpus")
except Exception:
    pass

coll = chroma.create_collection(name="day4_corpus")
embs = st_model.encode([c.text for c in corpus], show_progress_bar=False)
add_chunks(coll, corpus, embeddings=embs.tolist())
print(f"Indexed in Chroma: {coll.count()} chunks")

## 5. BM25 sparse retrieval

Dense embeddings are good for semantic similarity but weak on rare terms
(names, IDs, codes). BM25 is a bag-of-words ranker that's great for exact
keyword matches. Use both.

In [ ]:
from rank_bm25 import BM25Okapi

# Tokenize — simple whitespace split is fine for BM25
def tokenize(text):
    return re.findall(r"\w+", text.lower())


tokenized_corpus = [tokenize(c.text) for c in corpus]
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 index built.")


def bm25_search(query: str, k: int = 10) -> list[tuple[int, float]]:
    scores = bm25.get_scores(tokenize(query))
    top_idx = np.argsort(scores)[::-1][:k]
    return [(int(i), float(scores[i])) for i in top_idx]


def dense_search(query: str, k: int = 10) -> list[tuple[int, float]]:
    q_vec = st_model.encode([query])[0].tolist()
    res = coll.query(query_embeddings=[q_vec], n_results=k)
    # Map back to corpus index by id
    id_to_idx = {c.id: i for i, c in enumerate(corpus)}
    out = []
    for ret_id, dist in zip(res["ids"][0], res["distances"][0]):
        idx = id_to_idx.get(ret_id)
        if idx is not None:
            # Convert distance to similarity score (Chroma returns cosine distance)
            out.append((idx, 1 - float(dist)))
    return out


# Smoke test
q = "pep 8 indentation"
print(f"\nQuery: {q!r}")
print("BM25 top 3:")
for i, s in bm25_search(q, 3):
    print(f"  [{s:.2f}] {corpus[i].source}: {corpus[i].text[:80]}...")
print("Dense top 3:")
for i, s in dense_search(q, 3):
    print(f"  [{s:.2f}] {corpus[i].source}: {corpus[i].text[:80]}...")

## 6. Reciprocal Rank Fusion (RRF)

How do we combine BM25 scores and dense similarity scores? Their scales are
different, so averaging is wrong. RRF sidesteps this: only use *ranks*.

For each document `d`, RRF score = Σ 1 / (k_rrf + rank_in_list)
where k_rrf ≈ 60 is a smoothing constant.

In [ ]:
def reciprocal_rank_fusion(
    result_lists: list[list[tuple[int, float]]],
    k_rrf: int = 60,
) -> list[tuple[int, float]]:
    scores: dict[int, float] = {}
    for result_list in result_lists:
        for rank, (idx, _) in enumerate(result_list):
            scores[idx] = scores.get(idx, 0.0) + 1 / (k_rrf + rank)
    return sorted(scores.items(), key=lambda x: -x[1])


def hybrid_search(query: str, k: int = 10) -> list[tuple[int, float]]:
    dense = dense_search(query, k=k * 2)
    sparse = bm25_search(query, k=k * 2)
    fused = reciprocal_rank_fusion([dense, sparse])
    return fused[:k]


print(f"\nHybrid top 3 for {q!r}:")
for i, s in hybrid_search(q, 3):
    print(f"  [RRF={s:.3f}] {corpus[i].source}: {corpus[i].text[:80]}...")

## 7. Reranking with a cross-encoder

Bi-encoders (what we've been using) embed query and document *separately* —
fast but loses query-document interaction signal. Cross-encoders embed them
*together*, which is slower but much more accurate.

Strategy: use hybrid search to get top-20 candidates, then rerank those with
a cross-encoder to pick the final top-k. This is what Cohere Rerank does,
but we'll use a free local model.

In [ ]:
from sentence_transformers import CrossEncoder

# bge-reranker-base is ~280MB, first load takes 30s
reranker = CrossEncoder("BAAI/bge-reranker-base")


def rerank(query: str, candidate_indices: list[int], top_k: int = 5) -> list[tuple[int, float]]:
    pairs = [[query, corpus[i].text] for i in candidate_indices]
    scores = reranker.predict(pairs)
    scored = list(zip(candidate_indices, scores.tolist()))
    scored.sort(key=lambda x: -x[1])
    return scored[:top_k]


def retrieve_v2(query: str, final_k: int = 5, candidates: int = 20) -> list[dict]:
    hybrid_results = hybrid_search(query, k=candidates)
    candidate_idx = [i for i, _ in hybrid_results]
    reranked = rerank(query, candidate_idx, top_k=final_k)
    return [
        {
            "rank": r + 1,
            "score": score,
            "chunk": corpus[i],
        }
        for r, (i, score) in enumerate(reranked)
    ]


print(f"\nFull pipeline (hybrid → rerank) for {q!r}:")
for h in retrieve_v2(q):
    print(f"  rank {h['rank']} [score={h['score']:.2f}] {h['chunk'].source}")
    print(f"    {h['chunk'].text[:100]}...")

## 8. Measure: does hybrid + rerank actually beat dense-only?

We'll use the Day 3 PEP eval set to measure. We expect ~10–25% recall@5 lift
from hybrid + rerank, especially on queries with rare terms.

In [ ]:
eval_set = [
    {"q": "What indentation does PEP 8 recommend?",                              "source": "pep-0008.md"},
    {"q": "What is the maximum line length per PEP 8?",                          "source": "pep-0008.md"},
    {"q": "What does the Zen of Python say about explicit vs implicit?",         "source": "pep-0020.md"},
    {"q": "How should one-line docstrings be formatted?",                        "source": "pep-0257.md"},
    {"q": "What are the rules for multi-line docstrings?",                       "source": "pep-0257.md"},
    {"q": "What is an absolute import?",                                          "source": "pep-0328.md"},
    {"q": "Why were relative imports deprecated at one point?",                  "source": "pep-0328.md"},
    {"q": "What are type hints?",                                                 "source": "pep-0484.md"},
    {"q": "How does PEP 484 handle generic types?",                              "source": "pep-0484.md"},
    {"q": "How should imports be grouped?",                                       "source": "pep-0008.md"},
]


def recall_for_retriever(retriever_fn, name: str, k: int = 5) -> float:
    hits = 0
    for item in eval_set:
        results = retriever_fn(item["q"], k)
        sources = [corpus[i].source for i, _ in results]
        if item["source"] in sources:
            hits += 1
    r = hits / len(eval_set)
    print(f"  {name:25s} recall@{k} = {r:.2f}")
    return r


print("Retriever comparison:")
recall_for_retriever(dense_search, "Dense only")
recall_for_retriever(bm25_search, "BM25 only")
recall_for_retriever(hybrid_search, "Hybrid (RRF)")


def retrieve_v2_tuples(q, k):
    return [(h["chunk"].id, h["score"]) for h in retrieve_v2(q, final_k=k)]


# For rerank we need idx back — adapt
def rerank_recall(k=5):
    hits = 0
    for item in eval_set:
        results = retrieve_v2(item["q"], final_k=k, candidates=20)
        sources = [h["chunk"].source for h in results]
        if item["source"] in sources:
            hits += 1
    r = hits / len(eval_set)
    print(f"  {'Hybrid + Rerank':25s} recall@{k} = {r:.2f}")
    return r


rerank_recall()

## 9. Generation quality — Ragas metrics

Recall@k tells us whether we retrieved the right *document*. It doesn't tell
us whether the *generated answer* is any good. For that we need:

- **Faithfulness**: does the answer stick to the retrieved context? (No hallucination.)
- **Answer relevance**: does the answer actually address the question?
- **Context precision**: were the retrieved chunks actually useful?

Ragas provides these out of the box. It calls an LLM as judge — so it
costs money and has its own biases — but it's a starting point.

In [ ]:
def answer_with_sources(question: str) -> dict:
    hits = retrieve_v2(question, final_k=4)
    contexts = [h["chunk"].text for h in hits]
    sources_text = "\n\n".join(f"[Source {i+1}]\n{c}" for i, c in enumerate(contexts))
    prompt = (
        f"Sources:\n{sources_text}\n\n"
        f"Question: {question}\n\n"
        "Answer using only the sources above. Be concise (3–5 sentences)."
    )
    ans = generate(
        system="You are a precise assistant. Ground all answers in the provided sources.",
        user=prompt,
    )
    return {"question": question, "answer": ans, "contexts": contexts}


# Build a small eval dataframe
import json

eval_samples = []
for item in eval_set[:5]:  # small subset — Ragas is expensive
    r = answer_with_sources(item["q"])
    eval_samples.append({
        "user_input": r["question"],
        "response": r["answer"],
        "retrieved_contexts": r["contexts"],
    })

print(f"Generated {len(eval_samples)} samples for Ragas eval")
print(f"Sample:\n  Q: {eval_samples[0]['user_input']}")
print(f"  A: {eval_samples[0]['response'][:200]}...")

### Running Ragas

Ragas >= 0.2 uses the `EvaluationDataset` + `evaluate()` API. The snippet
below is the canonical shape; adjust model names to what your account has.

In [ ]:
# Run only if you're ready for a few API calls per metric per question.
# Uncomment when ready.

# from ragas import EvaluationDataset, evaluate
# from ragas.metrics import Faithfulness, ResponseRelevancy
# from ragas.llms import LangchainLLMWrapper
# from langchain_anthropic import ChatAnthropic  # or langchain_openai
#
# judge = LangchainLLMWrapper(ChatAnthropic(model="claude-haiku-4-5-20251001"))
# dataset = EvaluationDataset.from_list(eval_samples)
# result = evaluate(
#     dataset=dataset,
#     metrics=[Faithfulness(), ResponseRelevancy()],
#     llm=judge,
# )
# print(result)

print("See the commented block above — uncomment when your Ragas install is ready.")

## 10. Reflection

**By now your pipeline has real retrieval quality.** Things to check:

1. Did hybrid beat dense on *every* query or just some? Look at the ones
   where dense-only did better — what was different about them?
2. Reranking adds ~100ms of latency per query. Is it worth it for your
   client's use case?
3. Ragas faithfulness below 0.8 means Claude is hallucinating. Common causes:
   generation prompt too permissive, context too fragmented, question
   ambiguous. Diagnose before you tune.
4. Your scraping pipeline works on one source. A real engagement has 5–10.
   How do you generalize without writing bespoke parsers for each?

Tomorrow: real client data, full pipeline, production checklist, demos.